# 2. Training the generator

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chingkheinganba231005/Meitei-Mayek-Diffusion-model/blob/main/notebooks/2_train.ipynb)

**Runtime: A100.** One run of 32 GPU-hours, over at least two sessions.

1. **Checks** (about 10 minutes):
   - the overfit check: a small generator memorises 8 words and must give them back from pure
     noise (it prints `"passed": true`; if not, stop and find the cause);
   - the smoke test: memory, speed and 350 steps of the real model, and a sample sheet (noise
     at this stage: it checks the plumbing). It prints the steps 32 hours would give. With
     fewer than 80,000 (more than 1.4 s a step), set `CHANNELS = [96, 192, 384]` in the first
     cell before the main run. **Send these numbers before starting the main run.**
2. **The main run.** Set `START_MAIN_RUN = True` in the first cell (after buying the extra
   compute units), then run the first two cells and this one. Its length in steps is fixed from
   the speed of its first steps. A checkpoint goes to Drive every 20 minutes with a sample sheet
   of validation writers. If the runtime disconnects, or the session reaches 23.5 hours, run the
   first two cells and this one again: it resumes (at most 20 minutes are lost). Never start a
   second main run, and never change the model once it has started.
3. **Losses** from the run's history.

Inputs: `WORK/data` (notebook 1). Outputs: `WORK/runs/main/` (`last.pt`, about 2.7 GB;
`final.pt` with the three averages of the weights; snapshots `snap_*.pt`; `samples/`;
`history.json`) and `WORK/results/smoke.json`.

In [ ]:
import os

# Where things are. Change these to match your Drive.
WORK = "/content/drive/MyDrive/meitei-word-diffusion"         # this project's working folder
WORK_OLD = "/content/drive/MyDrive/meitei-word-recognition"   # the word-recognition project's folder
REPO = "chingkheinganba231005/Meitei-Mayek-Diffusion-model"
BRANCH = "main"
WORD_REPO = "chingkheinganba231005/meitei-mayek-word-recognition"
WORD_COMMIT = "bdee9e7"                                       # the synthesiser this project builds on
CONTENT = "/content"                                          # the runtime's own disk
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # before torch starts

HOURS = 32                      # GPU hours of the main run, over all sessions
CHANNELS = [128, 256, 512]      # the UNet's levels; [96, 192, 384] if the smoke test says so
START_MAIN_RUN = False          # True once the checks are fine

DISCONNECT_AT_END = True    # the last cell disconnects the runtime (Colab bills a connected runtime)

In [ ]:
import json, re, shutil, subprocess, sys, time
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
os.chdir(CONTENT)
CODE, WORD = f"{CONTENT}/code", f"{CONTENT}/word_repo"


def run(*args):
    """Runs a command, shows its output, and stops the notebook if it fails."""
    r = subprocess.run([str(a) for a in args], capture_output=True, text=True)
    if (r.stdout + r.stderr).strip():
        print(r.stdout + r.stderr)
    if r.returncode:
        raise SystemExit(f"exit code {r.returncode}: {' '.join(str(a) for a in args)[:200]}")


def stream(*args):
    """Runs a long command, showing its output as it comes; stops the notebook if it fails.
    Returns the output."""
    p = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        print(line, end="")
        lines.append(line)
    if p.wait():
        raise SystemExit(f"exit code {p.returncode}: {' '.join(str(a) for a in args)[:200]}")
    return "".join(lines)


def clone(url, path, ref):
    """A fresh copy of a repository at a branch or a commit."""
    shutil.rmtree(path, ignore_errors=True)
    run("git", "clone", "-q", url, path)
    run("git", "-C", path, "checkout", "-q", ref)


clone(f"https://github.com/{REPO}.git", CODE, BRANCH)
clone(f"https://github.com/{WORD_REPO}.git", WORD, WORD_COMMIT)
sys.path[:0] = [CODE, WORD]
os.chdir(CODE)

run(sys.executable, "-m", "pip", "install", "-q", "diffusers==0.41.0", "safetensors", "huggingface_hub", "timm")

for d in ("data", "runs", "results", "eval"):
    os.makedirs(f"{WORK}/{d}", exist_ok=True)
run("git", "-C", CODE, "log", "-1", "--format=code: %h %s")

run("nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader")

from IPython.display import Image as Show, display
MAIN, SMOKE = f"{WORK}/runs/main", f"{WORK}/runs/smoke"
if not os.path.exists(f"{WORK}/data/train/info.json"):
    raise SystemExit("WORK/data/train is not rendered yet: run notebook 1 first")

## 1. Checks

The overfit check (about a minute), then the smoke test (a few minutes, most of it loading the
training words). The smoke run is deleted at the end.

In [ ]:
out = stream(sys.executable, "-u", "scripts/overfit_check.py", "--word-repo", WORD, "--data", f"{WORK}/data",
             "--out", f"{WORK}/runs/overfit.png")
display(Show(f"{WORK}/runs/overfit.png"))   # rows: real | Heun | DPM-Solver++
if '"passed": true' not in out:
    raise SystemExit("the overfit check failed: do not start the main run")

shutil.rmtree(SMOKE, ignore_errors=True)
out = stream(sys.executable, "-u", "scripts/train.py", "--word-repo", WORD, "--data", f"{WORK}/data", "--run", SMOKE,
             "--local", f"{CONTENT}/ckpt_smoke", "--hours", 0.03, "--channels", *CHANNELS)
fits = re.search(r"a step fits in memory in (\d+) part\(s\); peak ([\d.]+) GB", out)
speed = re.search(r"([\d.]+) s a step: (\d+) steps in", out)
sec = float(speed.group(1))
steps = int(HOURS * 3600 / sec * 0.97)
smoke = {"channels": CHANNELS, "micro_batches": int(fits.group(1)) if fits else None,
         "peak_gb": float(fits.group(2)) if fits else None, "seconds_per_step": sec, "steps_in_32h": steps,
         "gpu": subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                               capture_output=True, text=True).stdout.strip()}
smoke["decision"] = ("keep the model" if steps >= 80000 else
                     "fewer than 80,000 steps: set CHANNELS = [96, 192, 384] before the main run")
Path(f"{WORK}/results/smoke.json").write_text(json.dumps(smoke, indent=1))
print(json.dumps(smoke, indent=1))
display(Show(f"{SMOKE}/samples/final.png"))
shutil.rmtree(SMOKE, ignore_errors=True)

## 2. The main run

About 32 hours. Re-run the first two cells and this one after any disconnection: it resumes.
The latest sample sheet is shown at the end (per validation writer: its four references, then
two of its words as written, left, and as generated, right).

In [ ]:
if not START_MAIN_RUN:
    raise SystemExit("Set START_MAIN_RUN = True in the first cell once the checks are fine, then run the first "
                     "two cells and this one.")
stream(sys.executable, "-u", "scripts/train.py", "--word-repo", WORD, "--data", f"{WORK}/data", "--run", MAIN,
       "--local", f"{CONTENT}/ckpt", "--hours", HOURS, "--channels", *CHANNELS)
sheets = sorted(Path(f"{MAIN}/samples").glob("*.png"), key=os.path.getmtime)
if sheets:
    print(sheets[-1].name)
    display(Show(str(sheets[-1])))

## 3. Losses

In [ ]:
import matplotlib.pyplot as plt

history = json.loads(Path(f"{MAIN}/history.json").read_text())
train_rows = [r for r in history if "loss" in r]
val_rows = [r for r in history if "val_wmse" in r]
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
for k in ("loss", "wmse"):
    ax[0].plot([r["step"] for r in train_rows], [r[k] for r in train_rows], label=k)
ax[0].set_yscale("log"); ax[0].legend(); ax[0].set_xlabel("step")
for k in ("supcon", "width"):
    ax[1].plot([r["step"] for r in train_rows], [r[k] for r in train_rows], label=k)
ax[1].set_yscale("log"); ax[1].legend(); ax[1].set_xlabel("step")
for k, label in (("val_wmse", "raw"), ("val_wmse_ema", "average 0.10")):
    ax[2].plot([r["step"] for r in val_rows], [r[k] for r in val_rows], label=label)
ax[2].set_title("validation weighted MSE"); ax[2].legend(); ax[2].set_xlabel("step")
fig.tight_layout()
fig.savefig(f"{MAIN}/losses.png", dpi=120)
plt.show()

## Disconnect

Colab bills a connected runtime whether or not it is working. The cell below disconnects it
(`DISCONNECT_AT_END = True`); otherwise use *Runtime > Disconnect and delete runtime*. Every
result is already on Drive.

In [ ]:
# Colab bills a connected runtime whether or not it is working: disconnect it now.
if DISCONNECT_AT_END:
    from google.colab import runtime
    runtime.unassign()
else:
    print("Finished. Disconnect the runtime: Runtime > Disconnect and delete runtime.")